In [1]:
%pip install -q pandas sentence-transformers bertopic hdbscan umap-learn transformers torch

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import re
import string
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root

titles_file_path = PROJECT_ROOT / "data" / "processed" / "titles.csv"
lyrics_file_path = PROJECT_ROOT / "data" / "processed" / "lyrics.csv"

analysis_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "lyrics_in_en.csv")

print(f'Loaded {len(analysis_df)} songs')
print(f'Columns: {list(analysis_df.columns)}')
analysis_df.head()

Loaded 754 songs
Columns: ['rank', 'artist', 'title', 'region', 'spotify_uri', 'lyrics_in_en', 'original_lang']


,rank,artist,title,region,spotify_uri,lyrics_in_en,original_lang
0,1,"Mr Plata, El Americano 4KT",Las Muñequitas,Colombia,4nJJCRYru4QQakCiUA155f,"(Tell me, are you going to give me what I ask ...",es
1,2,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),Colombia,3CBEVPwR3kUXDoTx1lqFUQ,"ARIA VEGA, Ryan Castro\nThe premium costñita a...",es
2,3,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,2ZyrAym0sRLwt4PhGotHuI,"Kapo, Ryan Castro, Gangsta\nWhat a joke, SOG\n...",es
3,4,Kris R.,GANAS,Colombia,4KE9Ne3hgh18B3Th4xcylg,"Yeah, yeah\nYeah, yeah\n\nMy love, let's fuck ...",es
4,5,"W Sound, Beéle, Ovy On The Drums",La Plena - W Sound 05,Colombia,6iOndD4OFo7GkaDypWQIou,O-O-Ovy On The Drums\nYou're the apple of my e...,en


### 1) Language distribution by region

In [8]:
lang_region = pd.crosstab(analysis_df['region'], analysis_df['original_lang'])
print(lang_region.to_string())

# Percentage view
lang_region_pct = pd.crosstab(analysis_df['region'], analysis_df['original_lang'], normalize='index').round(3) * 100
print('\nPercentage:')
print(lang_region_pct.to_string())

original_lang  ar  de   en   es  fr  gd  he  id  it  ja  ko  pt  ru  tr  unknown  vi  zh
region                                                                                  
Colombia        0   0   10  180   0   0   1   0   0   0   0   0   0   0        8   0   0
Global          1   1  137   23   0   0   0   3   1   0   0   3   2   4        3   0   0
Taiwan          1   0   46    0   0   2   0   0   1   2  24   0   1   0       50   1  64
USA             0   2  162    9   2   0   0   0   0   1   0   3   0   4        2   0   0

Percentage:
original_lang   ar   de    en    es   fr   gd   he   id   it   ja    ko   pt   ru   tr  unknown   vi    zh
region                                                                                                    
Colombia       0.0  0.0   5.0  90.5  0.0  0.0  0.5  0.0  0.0  0.0   0.0  0.0  0.0  0.0      4.0  0.0   0.0
Global         0.6  0.6  77.0  12.9  0.0  0.0  0.0  1.7  0.6  0.0   0.0  1.7  1.1  2.2      1.7  0.0   0.0
Taiwan         0.5  0.0  

### 2) Clean lyrics

In [10]:
def clean_lyrics(text: str) -> str:
    """Strip section tags, lowercase, remove punctuation, collapse whitespace."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = re.sub(r'\[[^\]]*\]', ' ', text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text




analysis_df['clean_lyrics'] = analysis_df['lyrics_in_en'].fillna('').map(clean_lyrics)

# Drop songs with no lyrics
has_lyrics = analysis_df['clean_lyrics'].str.strip().ne('')
print(f'Songs with lyrics: {has_lyrics.sum()} / {len(analysis_df)}')
df_valid = analysis_df[has_lyrics].reset_index(drop=True)
print(f'Proceeding with {len(df_valid)} songs')

Songs with lyrics: 691 / 754
Proceeding with 691 songs


### 3) Sentence embeddings + BERTopic

In [ ]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(
    df_valid['clean_lyrics'].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f'Embeddings shape: {embeddings.shape}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3773.60it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 22/22 [00:07<00:00,  3.12it/s]

Embeddings shape: (691, 384)


In [29]:
from sklearn.feature_extraction.text import CountVectorizer

n_samples = len(df_valid)
n_neighbors = max(2, min(6, n_samples - 1))
min_cluster_size = max(2, min(3, n_samples))
min_samples_val = max(1, min(5, n_samples))

umap_model = UMAP(
    n_neighbors=n_neighbors,
    n_components=5,
    min_dist=0.1,
    metric='cosine',
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=min_cluster_size,
    min_samples=min_samples_val,
    metric='euclidean',
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),   # capture phrases
    min_df=2              # ignore rare words
)

topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    min_topic_size=5
)

topics, probs = topic_model.fit_transform(
    df_valid['clean_lyrics'].tolist(), embeddings
)
print(f'Found {len(set(topics)) - (1 if -1 in topics else 0)} topics (+ outlier topic -1)')

Found 38 topics (+ outlier topic -1)


In [30]:
# Topic assignments
df_valid = df_valid.copy()
df_valid['topic'] = topics
df_valid['topic_probability'] = [
    float(np.max(p)) if isinstance(p, np.ndarray) and p.size > 0 else np.nan
    for p in probs
]

# Topic keywords
topic_keywords_rows = []
for tid in sorted(t for t in set(topics) if t != -1):
    words_scores = topic_model.get_topic(tid) or []
    topic_keywords_rows.append({
        'topic': tid,
        'keywords': ', '.join(w for w, _ in words_scores[:10]),
    })
topic_keywords = pd.DataFrame(topic_keywords_rows)

print('Topic keywords:')
print(topic_keywords.to_string(index=False))

Topic keywords:
 topic                                                                                                    keywords
     0                                                    love, want, dont, end, just, say, know, let, day, forget
     1                                          fashion, bitches, hey, fuck, nigga, im, hey hey, huh, lights, dont
     2                                             im, wanna, dishes, man, ima, em, know know, breakin, lose, know
     3                                             got, girls, ayy, tryin, im, thats like, thinkin, girl, like, il
     4                              body, baby, oh, ima, tonight, blackpink, music, blackpink blackpink, dj, wanna
     5                                okay okay, know, okay, wanna know, love, sea, kiss, know know, wanna, vision
     6                                           ooh, yeah, ooh ooh, yeah yeah, eat, eat eat, na, aw, really, like
     7                talk, dont talk, talk anymore, love love, 

In [23]:
pd.Series(topics).value_counts()

0    675
1     16
Name: count, dtype: int64

In [14]:
# Topic distribution by region
topic_by_region = pd.crosstab(
    df_valid['region'], df_valid['topic'], normalize='index'
).reset_index()
print('Topic distribution by region (%):')
print((topic_by_region.set_index('region') * 100).round(1).to_string())

Topic distribution by region (%):
topic         0    1
region              
Colombia  100.0  0.0
Global     98.9  1.1
Taiwan     90.8  9.2
USA        99.5  0.5


### 4) Emotion classification

In [15]:
from transformers import pipeline as hf_pipeline

emotion_clf = hf_pipeline(
    'text-classification',
    model='j-hartmann/emotion-english-distilroberta-base',
    return_all_scores=True,
)
print('Emotion classifier loaded.')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 60694.86it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Emotion classifier loaded.


In [17]:
def chunk_text(text: str, max_words: int = 180) -> list[str]:
    words = text.split()
    if not words:
        return ['']
    return [' '.join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

def emotion_scores(text: str) -> dict:
    chunks = chunk_text(text)
    all_scores = []
    for chunk in chunks:
        if not chunk.strip():
            continue
        raw = emotion_clf(chunk, truncation=True)
        if isinstance(raw, list) and raw and isinstance(raw[0], list):
            raw = raw[0]
        all_scores.append(raw)
    if not all_scores:
        return {}
    labels = sorted({item['label'] for scores in all_scores for item in scores})
    averaged = {}
    for label in labels:
        vals = [next((x['score'] for x in s if x['label'] == label), 0.0) for s in all_scores]
        averaged[label] = float(np.mean(vals))
    return averaged

print(f'Computing emotion scores for {len(df_valid)} songs...')
emo_series = df_valid['clean_lyrics'].map(emotion_scores)
emo_df = pd.json_normalize(emo_series).fillna(0.0)
emo_df.columns = [f'emotion_{c}' for c in emo_df.columns]

df_valid = pd.concat([df_valid.reset_index(drop=True), emo_df], axis=1)

emotion_cols = [c for c in df_valid.columns if c.startswith('emotion_')]
if emotion_cols:
    df_valid['dominant_emotion'] = (
        df_valid[emotion_cols].idxmax(axis=1).str.replace('emotion_', '', regex=False)
    )

print('Done.')
df_valid[['title', 'region', 'original_lang', 'dominant_emotion']].head(10)

Computing emotion scores for 691 songs...


KeyboardInterrupt: 

### 5) Cross-tabulations & save outputs

In [11]:
emotion_by_region = (
    df_valid.groupby('region')[emotion_cols].mean().reset_index()
    if emotion_cols else pd.DataFrame()
)
emotion_by_topic = (
    df_valid.groupby('topic')[emotion_cols].mean().reset_index()
    if emotion_cols else pd.DataFrame()
)
emotion_by_language = (
    df_valid.groupby('language')[emotion_cols].mean().reset_index()
    if emotion_cols else pd.DataFrame()
)

print('Emotion by region:')
print(emotion_by_region.to_string(index=False))
print('\nEmotion by topic:')
print(emotion_by_topic.to_string(index=False))
print('\nEmotion by language:')
print(emotion_by_language.to_string(index=False))

Emotion by region:
  region  emotion_anger  emotion_neutral  emotion_joy  emotion_fear  emotion_disgust  emotion_sadness  emotion_surprise
Colombia       0.026189         0.301945     0.048270      0.051681         0.007885         0.018186          0.009019
  Global       0.054750         0.106545     0.085603      0.067052         0.000000         0.221905          0.098847
  Taiwan       0.050664         0.179287     0.109355      0.045592         0.005671         0.161105          0.050961
     USA       0.072051         0.068157     0.088322      0.070707         0.001694         0.242703          0.093857

Emotion by topic:
 topic  emotion_anger  emotion_neutral  emotion_joy  emotion_fear  emotion_disgust  emotion_sadness  emotion_surprise
    -1       0.023955         0.233461     0.036903      0.115817         0.000000         0.076418          0.057561
     0       0.049864         0.000000     0.047730      0.103302         0.000000         0.362336          0.131999
     1  

In [12]:
run_id = f'lyrics_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
OUT_DIR = Path('..') / 'outputs' / run_id
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save all outputs
assignments_out = df_valid[[
    'title', 'artist', 'region', 'language', 'topic',
    'topic_probability', 'dominant_emotion'
] + emotion_cols]

assignments_out.to_csv(OUT_DIR / 'topic_assignments_per_song.csv', index=False)
topic_keywords.to_csv(OUT_DIR / 'topic_keywords.csv', index=False)
topic_by_region.to_csv(OUT_DIR / 'topic_distribution_by_region.csv', index=False)
emotion_by_region.to_csv(OUT_DIR / 'emotion_by_region.csv', index=False)
emotion_by_topic.to_csv(OUT_DIR / 'emotion_by_topic.csv', index=False)
emotion_by_language.to_csv(OUT_DIR / 'emotion_by_language.csv', index=False)

print(f'All outputs saved to {OUT_DIR}/')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')

All outputs saved to ../outputs/lyrics_20260311_164409/
  emotion_by_language.csv
  emotion_by_region.csv
  emotion_by_topic.csv
  topic_assignments_per_song.csv
  topic_distribution_by_region.csv
  topic_keywords.csv
